<!-- CONCLUSION-CELL -->
> ## ❌ 결론 — 도메인 feature(A 추세 + B 릴리스 변동성): **기각 (효과 없음)**
>
> | 단계 | features | val R² | test R² |
> |---|---|---|---|
> | E8-1 base | 59 | 0.0794 | 0.0294 |
> | E8-2 +B(변동성) | 65 | 0.0817 | 0.0327 |
> | E8-3 +A+B(전체) | 67 | 0.0865 | 0.0276 |
>
> **🔬 30 seed paired t-test (Val R²)**
> - base **0.0799** vs full **0.0799**, 차이 **-0.0000**
> - **t=-0.06, p=0.952 (≥0.05) → 유의미한 차이 없음 ❌** (full 승 15/30, 동전던지기)
> - 단계표의 val R² 상승은 seed=42 단판 노이즈 → 반복 시 소멸.
> - **→ A·B 도메인 feature 기각. 정형 baseline 유지.**


# 14. 도메인 Feature 확장 실험 (Phase 6.5 — A 추세 + B 릴리스 변동성)

**목적**: 15구 정적 평균만 쓰던 X feature에 도메인 논문 기반 feature를 추가해 예측력이 올라가는지 paired t-test로 검증.

**추가된 feature (8개, `feature_aggregator.py`에서 자동 생성)**

| 구분 | feature | 근거 논문 |
|---|---|---|
| **B 변동성** | `std_pos_x/z_{Fastball/Breaking/Offspeed}` (6개) | Frontiers — 수평 릴리스 변동성 ↔ 삼진 최강 상관 |
| **A 추세** | `trend_speed_all`, `trend_spin_all` (2개) | Paripex — 회전수/구속 저하 = 피로 1차 지표 |

**실험 설계**

| 실험 | 내용 | 비교 |
|---|---|---|
| E8-1 | 기존 feature만 (A/B 제외) | 기준 |
| E8-2 | + B 릴리스 변동성 | E8-1 vs E8-2 |
| E8-3 | + A 추세 (전체) | E8-2 vs E8-3 |
| 🔬 E8-4 | E8-1 vs E8-3 **paired t-test** (n=30 seeds) | 도메인 feature 기여도 검증 |
| E8-5 | SHAP으로 추세/변동성 feature 순위 확인 | delta보다 위로 올라오는지 |

> ⚠ **실행 전 필수**: `feature_aggregator.py`가 최신본(A/B feature 포함)이어야 함. 이 노트북은 `build_features`가 새 컴럼을 이미 내보낸다고 가정. 위에서 아래로 Run All 하면 됨.

In [ ]:
# ── 환경 감지 ──────────────────────────────
import os, sys

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/투수 컨디션 예측 ML'
    NB_DIR = os.path.join(DRIVE, '3_modeling')
    sys.path.insert(0, NB_DIR)
else:
    DRIVE  = os.path.dirname(os.path.abspath('__file__'))
    NB_DIR = os.path.dirname(os.path.abspath('__file__'))

INTERIM_DIR = os.path.join(DRIVE, '0_data', '2_interim')
FEATURE_DIR = os.path.join(DRIVE, '0_data', '4_features')
OUTPUT_DIR  = os.path.join(DRIVE, '4_output')
STARTERS    = os.path.join(INTERIM_DIR, 'starters_all.parquet')
LOOKUP      = os.path.join(INTERIM_DIR, 'prev_season_lookup.parquet')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# X구간 고정: pitch15
MODE = 'pitch'
N    = 15

print(f'환경: {"코랩" if IN_COLAB else "로컬"}')
print(f'X구간: {MODE}{N}')

In [ ]:
# ── 패키지 + 최적 파라미터 ────────────────────────
try:
    import duckdb, xgboost, shap
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'duckdb', 'xgboost', 'shap', '-q'])

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import xgboost as xgb
import shap
from sklearn.metrics import mean_squared_error, r2_score
import importlib
import feature_aggregator
importlib.reload(feature_aggregator)
from feature_aggregator import build_features

# Phase 7 튜닝 최적 파라미터 (best_params.json) 로드 — 없으면 기본값
BP_PATH = os.path.join(OUTPUT_DIR, 'best_params.json')
if os.path.exists(BP_PATH):
    XGB_PARAMS = json.load(open(BP_PATH, encoding='utf-8'))['XGB']
    print('best_params.json 로드:', XGB_PARAMS)
else:
    XGB_PARAMS = dict(n_estimators=500, learning_rate=0.05, max_depth=6,
                      subsample=0.8, colsample_bytree=0.8)
    print('best_params.json 없음 → 기본 파라미터 사용')

XGB_FIXED = dict(random_state=42, n_jobs=-1, verbosity=0, early_stopping_rounds=50)
print('패키지 로드 완료')

## 1. Feature 빌드 + A/B 컴럼 식별

`build_features`를 한 번 호출해 전체 feature를 만든 뒤, 이름 규칙으로 B/A 컴럼을 자동 분류한다.

In [ ]:
df = build_features(STARTERS, LOOKUP, MODE, N, include_delta=True)
print(f'전체: {len(df):,}행  |  컴럼 {len(df.columns)}개')

META_COLS = ['game_pk', 'pitcher', 'season', 'y_whiff', 'swings']
ALL_FEATS = [c for c in df.columns if c not in META_COLS]

# B 릴리스 변동성 / A 추세 컴럼 자동 식별
B_COLS = [c for c in ALL_FEATS if c.startswith('std_pos_')]
A_COLS = [c for c in ALL_FEATS if c.startswith('trend_')]
BASE_COLS = [c for c in ALL_FEATS if c not in B_COLS + A_COLS]

assert B_COLS, 'B(std_pos_*) 컴럼이 없음 → feature_aggregator.py가 구버전입니다. 최신본으로 교체 후 재실행.'
assert A_COLS, 'A(trend_*) 컴럼이 없음 → feature_aggregator.py 최신본으로 교체 후 재실행.'

print(f'\nBASE feature ({len(BASE_COLS)}개)')
print(f'B 변동성 ({len(B_COLS)}개): {B_COLS}')
print(f'A 추세 ({len(A_COLS)}개): {A_COLS}')

In [ ]:
# 시즌 기반 split (train 2021~2023 / val 2024 / test 2025)
train = df[df['season'].isin([2021, 2022, 2023])].copy()
val   = df[df['season'] == 2024].copy()
test  = df[df['season'] == 2025].copy()
print(f'train {len(train):,} | val {len(val):,} | test {len(test):,}')

def fit_eval(feat_cols, seed=42):
    """주어진 feature set으로 XGB 학습 → (val_r2, val_rmse, test_r2, model) 반환."""
    p = {**XGB_PARAMS, **XGB_FIXED, 'random_state': seed}
    m = xgb.XGBRegressor(**p)
    m.fit(train[feat_cols], train['y_whiff'],
          eval_set=[(val[feat_cols], val['y_whiff'])], verbose=False)
    vr2  = r2_score(val['y_whiff'], m.predict(val[feat_cols]))
    vrmse = mean_squared_error(val['y_whiff'], m.predict(val[feat_cols])) ** 0.5
    tr2  = r2_score(test['y_whiff'], m.predict(test[feat_cols])) if len(test) else float('nan')
    return vr2, vrmse, tr2, m

## 2. E8-1 ~ E8-3 단계적 추가 비교

In [ ]:
experiments = {
    'E8-1 base':       BASE_COLS,
    'E8-2 +B(변동성)':  BASE_COLS + B_COLS,
    'E8-3 +A+B(전체)':  BASE_COLS + B_COLS + A_COLS,
}

rows = []
for name, cols in experiments.items():
    vr2, vrmse, tr2, _ = fit_eval(cols)
    rows.append(dict(name=name, n_features=len(cols),
                     val_r2=round(vr2, 4), val_rmse=round(vrmse, 4),
                     test_r2=round(tr2, 4)))
    print(f'{name:18s} feat={len(cols):3d}  val_R²={vr2:.4f}  val_RMSE={vrmse:.4f}  test_R²={tr2:.4f}')

result_table = pd.DataFrame(rows)
result_table

## 3. 🔬 E8-4 Paired t-test — 기존(E8-1) vs 전체(E8-3)

30개 random seed로 반복 학습 → Val R² 쌍으로 paired t-test.

In [ ]:
N_SEEDS = 30
r2_base, r2_full = [], []

for i, seed in enumerate(range(N_SEEDS)):
    r2_base.append(fit_eval(BASE_COLS, seed)[0])
    r2_full.append(fit_eval(BASE_COLS + B_COLS + A_COLS, seed)[0])
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{N_SEEDS} 완료')

r2_base = np.array(r2_base)
r2_full = np.array(r2_full)
diff = r2_full - r2_base

t_stat, p_value = stats.ttest_rel(r2_full, r2_base)

print('\n' + '=' * 52)
print('Paired t-test: 도메인 feature(A+B) 기여도')
print('=' * 52)
print(f'E8-1 base   평균 R²: {r2_base.mean():.4f} ± {r2_base.std():.4f}')
print(f'E8-3 전체    평균 R²: {r2_full.mean():.4f} ± {r2_full.std():.4f}')
print(f'평균 차이 (전체 - base): {diff.mean():+.4f}')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value    : {p_value:.4f}')
print()
if p_value < 0.05:
    print('→ p < 0.05: 도메인 feature가 통계적으로 유의미한 개선 ✅')
else:
    print('→ p ≥ 0.05: 통계적으로 유의미한 차이 없음 ❌')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.boxplot([r2_base, r2_full], labels=['E8-1\nbase', 'E8-3\n+A+B'], patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('Val R²')
ax.set_title(f'R² 분포 비교 (n={N_SEEDS} seeds)\np={p_value:.4f}')
ax.grid(alpha=0.3)

ax = axes[1]
ax.hist(diff, bins=15, color='darkorange', alpha=0.7, edgecolor='white')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(diff.mean(), color='red', linewidth=1.5, label=f'mean={diff.mean():+.4f}')
ax.set_xlabel('R² 차이 (전체 - base)')
ax.set_ylabel('빈도')
ax.set_title('도메인 feature 기여 차이 분포')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'domain_feature_ttest.png'), dpi=120, bbox_inches='tight')
plt.show()

## 4. E8-5 SHAP — 추세/변동성 feature 순위 확인

delta feature가 하위권(최고 27위)였던 08번 한계를, 새 feature가 극복하는지 확인.

In [ ]:
FULL_COLS = BASE_COLS + B_COLS + A_COLS
_, _, _, model_full = fit_eval(FULL_COLS)

explainer = shap.TreeExplainer(model_full)
shap_vals = explainer(val[FULL_COLS])
shap_mean = pd.Series(np.abs(shap_vals.values).mean(axis=0),
                      index=FULL_COLS).sort_values(ascending=False)
rank = {f: i + 1 for i, f in enumerate(shap_mean.index)}

print(f'전체 {len(FULL_COLS)}개 feature 중 새 feature SHAP 순위:\n')
print('[B 릴리스 변동성]')
for f in sorted(B_COLS, key=lambda x: rank[x]):
    print(f'  {rank[f]:3d}위  {f:24s}  SHAP={shap_mean[f]:.5f}')
print('[A 추세]')
for f in sorted(A_COLS, key=lambda x: rank[x]):
    print(f'  {rank[f]:3d}위  {f:24s}  SHAP={shap_mean[f]:.5f}')

print('\n[상위 15개 전체]')
for i, (f, v) in enumerate(shap_mean.head(15).items()):
    tag = '  ← 새 feature' if (f in B_COLS or f in A_COLS) else ''
    print(f'  {i+1:3d}위  {f:24s}  {v:.5f}{tag}')

In [ ]:
# SHAP bar plot (상위 20개, 새 feature 강조)
top = shap_mean.head(20)[::-1]
colors = ['crimson' if (f in B_COLS or f in A_COLS) else 'steelblue' for f in top.index]
plt.figure(figsize=(9, 7))
plt.barh(top.index, top.values, color=colors, alpha=0.85)
plt.xlabel('mean |SHAP|')
plt.title('SHAP feature importance (top 20)\n빨간색 = 새 도메인 feature (A 추세 / B 변동성)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'domain_feature_shap.png'), dpi=120, bbox_inches='tight')
plt.show()

## 5. 결과 저장

In [ ]:
# 단계표 저장
result_table.to_csv(os.path.join(OUTPUT_DIR, 'domain_feature_results.csv'),
                    index=False, encoding='utf-8-sig')

# paired t-test seed별 + 요약 저장
pd.DataFrame({'seed': range(N_SEEDS), 'r2_base': r2_base,
              'r2_full': r2_full, 'diff': diff}).to_csv(
    os.path.join(OUTPUT_DIR, 'domain_feature_ttest_seeds.csv'),
    index=False, encoding='utf-8-sig')

summary = {
    'E8-1 base mean R²':  round(float(r2_base.mean()), 4),
    'E8-3 full mean R²':  round(float(r2_full.mean()), 4),
    'mean diff':          round(float(diff.mean()), 4),
    't_stat':             round(float(t_stat), 4),
    'p_value':            round(float(p_value), 4),
    'significant':        bool(p_value < 0.05),
}
json.dump(summary, open(os.path.join(OUTPUT_DIR, 'domain_feature_ttest.json'),
                        'w', encoding='utf-8'), ensure_ascii=False, indent=2)

print('저장 완료:')
print('  4_output/domain_feature_results.csv')
print('  4_output/domain_feature_ttest_seeds.csv')
print('  4_output/domain_feature_ttest.json')
print('  4_output/domain_feature_ttest.png')
print('  4_output/domain_feature_shap.png')
print('\n요약:', summary)